# Analyzing noJUmP Demo Results

This notebook analyzes the performance data exported by `run_demo.py`.

The export level depends on how the demo was run:

- **Direct run** (no SLURM): `user` level - files named `perf_data_user.*`
- **SLURM allocation**: `slurm` level - files named `perf_data_slurm.*`

The session directory (`session/`) always contains CSVs for all levels.

Adjust the paths and level below to match your run.

In [ ]:
import os
import glob

# Set this to "user" or "slurm" depending on how the demo was run
export_level = "slurm" if os.environ.get("SLURM_JOB_ID") else "user"

# Find the latest run_* directory
run_dirs = sorted(glob.glob("run_*"))
if run_dirs:
    run_dir = run_dirs[-1]
    print(f"Using run directory: {run_dir}")
else:
    run_dir = "."
    print("No run_* directory found, using current directory")
print(f"Using export level: {export_level}")

## Option 1: Import the full session

This restores the complete monitoring session including cell history, performance data, and hardware info. Requires the JUmPER extension.

In [ ]:
%load_ext jumper_extension

In [ ]:
session_path = os.path.join(run_dir, "session")
get_ipython().run_line_magic('import_session', session_path)

In [ ]:
%perfmonitor_resources

In [ ]:
%show_cell_history

In [ ]:
%perfmonitor_plot

In [ ]:
f"%perfmonitor_plot --level {export_level}"

## Option 2: Load the pickle file directly

The pickle file contains the raw performance DataFrames (not matplotlib
figures), so it is safe to load across different matplotlib versions.
The filename includes the export level.

In [ ]:
import pickle
import pandas as pd

pickle_path = os.path.join(run_dir, f"perf_data_{export_level}.pkl")
with open(pickle_path, "rb") as f:
    data = pickle.load(f)

# The pickle contains plain Python dicts (not DataFrames) for
# cross-version compatibility. Reconstruct DataFrames here.
if isinstance(data, dict):
    frames = {}
    for key, val in data.items():
        if isinstance(val, list):
            frames[key] = pd.DataFrame(val)
        else:
            frames[key] = val
    data = frames
elif isinstance(data, list):
    data = pd.DataFrame(data)

print(f"Loaded {pickle_path}")
print(type(data))
if isinstance(data, dict):
    for key, val in data.items():
        print(f"{key}: {type(val)}")
        if isinstance(val, pd.DataFrame):
            print(val.head())
elif isinstance(data, pd.DataFrame):
    print(data.head())
else:
    print(data)

## Option 3: Load the CSV directly

Two CSV sources are available:
- The level-specific CSV exported by the script (e.g. `perf_data_user.csv`)
- The per-level CSVs inside the session directory (e.g. `session/perf_user.csv`)

In [ ]:
csv_path = os.path.join(run_dir, f"perf_data_{export_level}.csv")
df = pd.read_csv(csv_path)
print(f"Loaded {csv_path} ({len(df)} rows)")
df.head()

In [ ]:
session_csv = os.path.join(run_dir, "session", f"perf_{export_level}.csv")
df_session = pd.read_csv(session_csv)
print(f"Loaded {session_csv} ({len(df_session)} rows)")
df_session.head()

In [ ]:
cell_history = pd.read_csv(os.path.join(run_dir, "session", "cell_history.csv"))
cell_history

## Summary

- Use **Option 1** (session import) for the full interactive JUmPER experience with plots and reports.
- Use **Option 2** (pickle) or **Option 3** (CSV) for lightweight, dependency-free analysis.
- The JPEG file (`perf_plot_<level>.jpeg`) provides a static visual snapshot of the performance data.